# 🐶 Dog Breed Classification — 120 Breeds (PRAICP-1007)
**Business case:** develop a machine-learning model that can **classify the breed of a dog from its image**.
**Project goal (from the brief):** create a model that classifies the dog breed on the basis of the image.
**Data (from the brief):** 120 directories, one per breed → a **120-class** image-classification problem.

### Requirement traceability (project brief → this notebook)
| Requirement in the project document | Where it is covered |
|---|---|
| Classify the breed of a dog | Whole notebook; final model predicts one of 120 breeds + top-5 list (§7) |
| Classify on the basis of the **image** | Image preprocessing & augmentation (§3), CNN / transfer-learning models (§4) |
| **120 directories = 120 breeds** | Folder → breed labelling with a hard check that 120 breeds are found (§2.1) |
| Dataset: `.../CNN-PROJECT-1-6/dog-breed-120.zip` | Automatic download + extraction (§1); or point `DATA_PATH` to an extracted copy |

.

## 1. Setup and dataset download

In [ ]:
import os, re, json, time, zipfile, hashlib, pathlib, urllib.request
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

# ---------------- configuration (everything you may want to change is here) ----------------
DATA_URL    = "https://d3ilbtxij3aepc.cloudfront.net/projects/CNN-PROJECT-1-6/dog-breed-120.zip"   # from the project brief
ZIP_PATH    = "dog-breed-120.zip"
EXTRACT_DIR = "dog-breed-120"
DATA_PATH   = None        # optional: path of an already-extracted folder, e.g. r"C:\data\dog-breed-120\Dogs class"

EXPECTED_BREEDS = 120
IMG_SIZE   = 224          # fine detail (ears, muzzle, coat) matters for breeds
BATCH_SIZE = 32
EPOCHS = dict(baseline=12, cnn1=20, cnn2=35, mob1=10, mob2=20, eff1=10, eff2=20)   # EarlyStopping stops earlier if val_loss stops improving
RUN    = dict(baseline=True, cnn1=True, cnn2=True, mobilenet=True, effnet=True)    # set a model to False to skip it (e.g. to save GPU time)
PRETRAINED_WEIGHTS = "imagenet"          # needs internet on first run
TRAIN_EVAL_N = 3000                      # size of the (un-augmented) train sample used for fair train-accuracy evaluation
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| GPUs:", gpus)
if not gpus:
    print("WARNING: no GPU found - training 120 classes at", IMG_SIZE, "px will be very slow. Use Colab (T4 GPU) or lower IMG_SIZE / switch off models in RUN.")

In [ ]:
def find_dataset_root(base):
    # Return the folder that has the most sub-folders containing images (= the folder holding the breed folders).
    best, best_n = None, 0
    for d, subdirs, _ in os.walk(base):
        n = 0
        for s in subdirs:
            try:
                if any(f.lower().endswith(tuple(IMG_EXTS)) for f in os.listdir(os.path.join(d, s))[:50]): n += 1
            except OSError:
                pass
        if n > best_n: best, best_n = d, n
    return best, best_n

if DATA_PATH and os.path.isdir(DATA_PATH):
    base = DATA_PATH
elif os.path.isdir(EXTRACT_DIR):
    base = EXTRACT_DIR
else:
    if not os.path.exists(ZIP_PATH):
        print("Downloading dataset ...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
    base = EXTRACT_DIR

path, n_dirs = find_dataset_root(base)
assert path is not None and n_dirs >= 2, f"No breed folders with images found under {base}"
print("Dataset root:", path)
print("Breed folders with images:", n_dirs)
print(sorted(os.listdir(path))[:8], "...")

### 💡 Insight
- The notebook now **downloads the dataset itself** (URL from the project brief) and **locates the folder that holds the breed folders** automatically, even if the zip contains an extra wrapper folder (such as `Dogs class`) or a `description.txt`. Nothing is hard-coded to one computer any more.
- The number of breed folders found here must be **120** — the next section checks it against the brief.

## 2. Exploratory Data Analysis (EDA)

### 2.1 Label the breed folders
Each folder is one breed. Folder names look like `n02085620-Chihuahua` (an ImageNet-style ID + name), so every name is converted into a clean, readable breed label (ID prefix removed, `_`/`-` → space, title case).

In [ ]:
SPLIT_NAMES = {"train", "training", "val", "valid", "validation", "test", "testing", "images", "image", "data", "dataset", "dogs"}

def clean_breed(folder_name):
    s = re.sub(r"^n\d{8}[-_ ]*", "", folder_name)       # ImageNet-style id prefix
    s = re.sub(r"^\d+[\.\-_ ]+", "", s)                # numeric prefix such as '001.'
    s = s.replace("_", " ").replace("-", " ").strip()
    return s.title() if s else folder_name

rows = []
for p in pathlib.Path(path).rglob("*"):
    if p.suffix.lower() in IMG_EXTS and not p.name.startswith("."):
        folders = [x for x in p.parent.relative_to(path).parts if x.lower() not in SPLIT_NAMES]
        if folders: rows.append([str(p), folders[-1]])

df = pd.DataFrame(rows, columns=["image_path", "folder"])
assert len(df) > 0, "No images found - check `path`."
folder_to_breed = {f: clean_breed(f) for f in sorted(df["folder"].unique())}
if len(set(folder_to_breed.values())) < len(folder_to_breed):      # avoid label collisions
    folder_to_breed = {f: f for f in folder_to_breed}
df["label"] = df["folder"].map(folder_to_breed)

classes = sorted(df["label"].unique()); n_classes = len(classes)
print(f"{len(df)} images | {n_classes} breeds")
if n_classes == EXPECTED_BREEDS:
    print(f"OK: found exactly the {EXPECTED_BREEDS} breeds mentioned in the project brief.")
else:
    print(f"WARNING: the brief says {EXPECTED_BREEDS} breeds but {n_classes} were found - check the folder structure.")
display(pd.DataFrame({"folder": list(folder_to_breed.keys()), "breed_label": list(folder_to_breed.values())}).head(12))

### 💡 Insight
Labels come straight from the folder names, so there is **no manual labelling and no label-noise risk from the labelling step**. The check against 120 confirms that every breed folder in the project document was found. The label list (sorted alphabetically) defines the output neuron of every model.

### 2.2 Sample images

In [ ]:
picks = df.sample(24, random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(4, 6, figsize=(16, 11))
for ax, (_, r) in zip(axes.ravel(), picks.iterrows()):
    ax.imshow(Image.open(r["image_path"]).convert("RGB")); ax.set_title(r["label"], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

### 💡 Insight
Photos are **natural, uncontrolled images**: different poses, distances, backgrounds, lighting, sometimes people or several dogs in the frame, and the dog is often small in the picture. This is why the model needs (a) augmentation, and (b) a strong pretrained feature extractor — the breed is a *fine-grained* visual detail (ear shape, muzzle, coat colour and texture), not an obvious global shape.

### 2.3 Data quality: corrupt files, sizes, colour modes and **duplicate images**
Every file is opened once to (a) verify it is readable, (b) record its size and colour mode, and (c) compute a small **perceptual hash** (difference-hash). Identical pictures and **resized copies** get the same hash (heavily re-compressed copies can still slip through). If the same photo sits in both train and test the test score is inflated (**data leakage**), so duplicates are removed **before** splitting. A photo that appears under two different breeds would be a label error and is dropped completely.

In [ ]:
def scan(path_, size=16):
    try:
        with Image.open(path_) as im: im.verify()
        with Image.open(path_) as im:
            wh, mode = im.size, im.mode
            im.draft("L", (size * 8, size * 8))                      # fast JPEG decode at reduced size
            g = np.asarray(im.convert("L").resize((size + 1, size), Image.LANCZOS), dtype=np.int16)
        return wh[0], wh[1], mode, hashlib.md5((g[:, 1:] > g[:, :-1]).tobytes()).hexdigest()
    except Exception:
        return None

t0 = time.time()
res = [scan(p) for p in df["image_path"]]
print(f"Scanned {len(df)} images in {time.time() - t0:.0f}s")
bad = [i for i, r in enumerate(res) if r is None]
print(f"Corrupt / unreadable files: {len(bad)}")
df = df.drop(index=bad).reset_index(drop=True)
good = [r for r in res if r is not None]
df["w"], df["h"], df["mode"], df["hash"] = [r[0] for r in good], [r[1] for r in good], [r[2] for r in good], [r[3] for r in good]

print("Colour modes:", dict(Counter(df["mode"])))
print(f"Width  min/median/max: {df.w.min()}/{int(df.w.median())}/{df.w.max()}")
print(f"Height min/median/max: {df.h.min()}/{int(df.h.median())}/{df.h.max()}")
print("Most common sizes:", Counter(zip(df.w, df.h)).most_common(3))

dup_mask = df.duplicated("hash", keep=False)
conflict = df.groupby("hash")["label"].transform("nunique") > 1
print(f"\nImages that have a duplicate / near-duplicate: {int(dup_mask.sum())} (in {df.loc[dup_mask, 'hash'].nunique()} groups)")
print(f"Duplicate groups with CONFLICTING breed labels: {df.loc[conflict, 'hash'].nunique()}")
n0 = len(df)
df = df[~conflict].drop_duplicates("hash", keep="first").reset_index(drop=True)
print(f"Images before: {n0} -> after removing duplicates/conflicts: {len(df)}  (removed {n0 - len(df)})")

### 💡 Insight
- **Image sizes vary widely** (from ~100 px to several thousand pixels), so everything is resized to one square size. Almost all images are RGB; any rare RGBA/CMYK/grayscale file is converted to RGB by the generators.
- Duplicates are a real risk in scraped/crowd-sourced image sets. The hash also catches **resized copies**, which a plain file-checksum would miss (heavily re-compressed copies may still slip through). Removing them keeps the validation and test scores honest.
- Remember this is *not* a guarantee against every kind of leakage — see the ImageNet caveat in section 5.

### 2.4 Class balance across the breeds

In [ ]:
class_counts = df["label"].value_counts()
chance_top1, chance_top5 = 1 / n_classes, min(5, n_classes) / n_classes
print("Images per breed -> min:", class_counts.min(), "| median:", int(class_counts.median()), "| max:", class_counts.max(),
      "| imbalance ratio (max/min):", round(class_counts.max() / class_counts.min(), 2))
print(f"Chance level: top-1 = {chance_top1:.2%}, top-5 = {chance_top5:.2%}  (what a model guessing at random would score)")

fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
ax[0].hist(class_counts.values, bins=20, color="steelblue", edgecolor="white")
ax[0].set_xlabel("Images per breed"); ax[0].set_ylabel("Number of breeds"); ax[0].set_title("Distribution of class sizes")
k = min(10, len(class_counts))
tb = pd.concat([class_counts.head(k), class_counts.tail(k)])
ax[1].barh(tb.index[::-1], tb.values[::-1], color=["#d6619a"] * k + ["steelblue"] * k)
ax[1].set_title(f"{k} largest (blue) and {k} smallest (pink) breeds"); ax[1].set_xlabel("Images")
plt.tight_layout(); plt.show()
assert class_counts.min() >= 3, "A breed has fewer than 3 images - a stratified split is not possible."
print(f"INSIGHT: a typical breed has {int(class_counts.median())} images; the largest breed has {class_counts.max() / class_counts.min():.1f}x as many images as the smallest.")

### 💡 Insight
- A few hundred images per breed at most is **small for 120 fine-grained classes** — a CNN trained from scratch will overfit quickly. That is the main argument for transfer learning.
- The imbalance is **moderate**; we still use balanced **class weights** so small breeds are not neglected, and we report **per-breed** metrics (a high overall accuracy can hide weak breeds).
- **Chance level** (≈ 1 / number of breeds for top-1) is the reference for the weakest models: the Dense baseline should be above it, but not by a lot.

### 2.5 Breed families — which breeds look alike?

In [ ]:
FAMILIES = ["terrier", "spaniel", "retriever", "hound", "poodle", "schnauzer", "setter", "shepherd", "collie", "sheepdog", "bulldog", "pinscher", "spitz", "mastiff"]
def family_of(label):
    low = label.lower()
    for f in FAMILIES:
        if f in low: return f
    return "other"

fam = pd.Series({b: family_of(b) for b in classes})
fam_counts = fam[fam != "other"].value_counts()
print(f"{(fam != 'other').sum()} of {n_classes} breeds belong to a name-based family (terrier, spaniel, ...); {(fam == 'other').sum()} are 'other'.")
plt.figure(figsize=(9, 4)); fam_counts.plot(kind="bar", color="slateblue"); plt.ylabel("number of breeds"); plt.title("Breeds that share a family name"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

### 💡 Insight
Many breeds share a family (e.g. several terriers, spaniels, hounds, retrievers) and differ only in **subtle details** — colour pattern, ear carriage, coat length. This is a **fine-grained** recognition task: confusions inside a family are expected, and are much less serious than confusing a terrier with a retriever. In the error analysis (section 6) we measure exactly how many errors stay *inside* a family.

### 2.6 Image size and shape distribution

In [ ]:
df["aspect"] = df["w"] / df["h"]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
ax[0].hist(df["w"].clip(upper=1000), bins=40, color="teal"); ax[0].set_title("Width (px, clipped at 1000)")
ax[1].hist(df["h"].clip(upper=1000), bins=40, color="darkorange"); ax[1].set_title("Height (px, clipped at 1000)")
ax[2].hist(df["aspect"].clip(0.4, 2.5), bins=40, color="crimson"); ax[2].set_title("Aspect ratio width / height"); ax[2].axvline(1, color="k", ls="--")
plt.tight_layout(); plt.show()
small = (np.minimum(df.w, df.h) < IMG_SIZE).mean(); wide = ((df.aspect > 1.2) | (df.aspect < 0.83)).mean()
print(f"INSIGHT: {small:.1%} of images have a side shorter than {IMG_SIZE}px (they get up-scaled); {wide:.1%} are clearly non-square (they get squeezed to a square).")

### 💡 Insight
Most photos are landscape/portrait rather than square, so **resizing to a square distorts the aspect ratio** slightly. This is normal in image classification and the networks cope with it, but it is one reason why a *dog detector + crop* step (see next steps) usually adds accuracy. Very small source images are up-scaled; they carry less detail and tend to be among the hardest.

### 2.7 Intra-breed variety vs inter-breed similarity

In [ ]:
fam_for_plot = "terrier" if (fam == "terrier").sum() >= 3 else (fam_counts.index[0] if len(fam_counts) else None)
plot_breeds = [b for b in classes if fam.get(b) == fam_for_plot][:4] if fam_for_plot else classes[:4]
fig, axes = plt.subplots(len(plot_breeds), 5, figsize=(14, 2.9 * len(plot_breeds)))
axes = np.atleast_2d(axes)
for r_, b in enumerate(plot_breeds):
    for c_, p in enumerate(df[df.label == b].sample(5, random_state=SEED)["image_path"]):
        axes[r_, c_].imshow(Image.open(p).convert("RGB")); axes[r_, c_].axis("off")
        if c_ == 0: axes[r_, c_].set_title(b, fontsize=10, loc="left")
plt.suptitle(f"Five random photos each of {len(plot_breeds)} breeds from the '{fam_for_plot}' family", y=1.0); plt.tight_layout(); plt.show()

### 💡 Insight
Look at one row: photos of the **same breed** differ a lot (pose, background, age, coat colour). Look across the rows: breeds of the **same family** look similar. A classifier has to be *invariant* to nuisance factors within a breed and *sensitive* to small differences between breeds — which is why generic features learned from millions of ImageNet images help so much.

### 2.8 Shuffle and split: Train / Validation / Test = 70 / 15 / 15 (stratified by breed)

In [ ]:
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
train_data, temp_data = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["label"])
val_data, test_data   = train_test_split(temp_data, test_size=0.50, random_state=SEED, stratify=temp_data["label"])
train_data, val_data, test_data = [d.reset_index(drop=True) for d in (train_data, val_data, test_data)]
print(f"Train: {len(train_data)} | Validation: {len(val_data)} | Test: {len(test_data)}")
per_breed_test = test_data["label"].value_counts()
print(f"Test images per breed: min {per_breed_test.min()}, median {int(per_breed_test.median())}, max {per_breed_test.max()}")

### 💡 Insight
Stratified splitting keeps **every breed present in all three sets** with the same proportion. With ~20 test images per breed, the *per-breed* accuracy is a coarse estimate (one extra mistake moves it by ≈ 5 points), so look at breed-level results as indications, and at the overall accuracy (≈ 3 000 test images) for the reliable number.

## 3. Preprocessing
- **Resize** to 224 × 224 and **normalise** pixels to 0–1.
- **Augmentation on the training set only** (validation/test must stay untouched): flips, small rotations, shifts, shear, zoom and brightness changes are realistic for dog photos (different pose, distance, lighting) and do not change the breed.
- Three evaluation sets use *rescaling only*; an extra **un-augmented sample of the training images** is used to measure training accuracy fairly (the augmented training stream is intentionally harder and would understate it).

In [ ]:
train_datagen = ImageDataGenerator(rescale=1.0 / 255, rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
                                   shear_range=0.1, zoom_range=0.2, brightness_range=(0.8, 1.2), horizontal_flip=True, fill_mode="nearest")
plain_datagen = ImageDataGenerator(rescale=1.0 / 255)

def make_gen(datagen, frame, shuffle):
    return datagen.flow_from_dataframe(dataframe=frame, x_col="image_path", y_col="label", target_size=(IMG_SIZE, IMG_SIZE),
                                       batch_size=BATCH_SIZE, class_mode="categorical", shuffle=shuffle, seed=SEED)

train_eval_data = train_data.sample(min(TRAIN_EVAL_N, len(train_data)), random_state=SEED).reset_index(drop=True)
train_generator      = make_gen(train_datagen, train_data, True)
train_eval_generator = make_gen(plain_datagen, train_eval_data, False)
validation_generator = make_gen(plain_datagen, val_data, False)
test_generator       = make_gen(plain_datagen, test_data, False)

class_labels = [k for k, v in sorted(train_generator.class_indices.items(), key=lambda kv: kv[1])]
assert class_labels == classes and len(class_labels) == n_classes
for name, g in [("Train", train_generator), ("Validation", validation_generator), ("Test", test_generator)]:
    xb, yb = g[0]; print(f"{name:10s} batch {xb.shape}, labels {yb.shape}, batches/epoch {len(g)}, pixel range [{xb.min():.2f}, {xb.max():.2f}]")

In [ ]:
images, labels = train_generator[0]
plt.figure(figsize=(16, 3.4))
for i in range(6):
    plt.subplot(1, 6, i + 1); plt.imshow(np.clip(images[i], 0, 1)); plt.title(class_labels[int(np.argmax(labels[i]))], fontsize=9); plt.axis("off")
plt.suptitle("Augmented training samples"); plt.show()

### 💡 Insight
Pixel values are in 0–1 as intended. The augmented samples should still show a clearly recognisable breed; each epoch the network sees a slightly different version of every training photo, which acts as a **regulariser** — important with only ~100–200 training images per breed.

## 4. Model Building
Five models are trained so the final choice is backed by evidence, each one answering a question:

| # | Model | Question it answers |
|---|---|---|
| 1 | **Dense baseline** (64×64, no convolutions) | How far do raw pixels get us? |
| 2 | **CNN v1** (plain, from scratch) | Do convolutions help? |
| 3 | **CNN v2** (BatchNorm + Dropout + L2 + GAP, from scratch) | Can regularisation rescue a from-scratch model? |
| 4 | **MobileNetV2** (ImageNet, 2 phases) | Does pretrained knowledge help? (small, fast) |
| 5 | **EfficientNetB0** (ImageNet, 2 phases) | Does a stronger backbone help more? |

All models use `EarlyStopping` and `ReduceLROnPlateau` on **`val_loss`** with best-weights restore, and **balanced class weights**. Accuracy is reported as **top-1** and **top-5** (is the true breed among the 5 most likely predictions?).

In [ ]:
cw = compute_class_weight("balanced", classes=np.arange(n_classes), y=train_generator.classes)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print(f"class weights: min {min(cw):.2f}, max {max(cw):.2f}")

def make_metrics():                         # fresh metric objects for every model (never shared)
    return ["accuracy", keras.metrics.TopKCategoricalAccuracy(k=min(5, n_classes), name="top5")]

def get_callbacks(patience):
    return [EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1)]

trained, HISTORIES, TRAIN_TIME = {}, {}, {}

def plot_history(hist, title):
    ep = range(1, len(hist["accuracy"]) + 1)
    plt.figure(figsize=(13, 4.5))
    plt.subplot(1, 2, 1); plt.plot(ep, hist["accuracy"], label="train (augmented)"); plt.plot(ep, hist["val_accuracy"], label="validation top-1"); plt.plot(ep, hist["val_top5"], label="validation top-5", ls="--")
    plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.title(f"Accuracy - {title}"); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(ep, hist["loss"], label="train"); plt.plot(ep, hist["val_loss"], label="validation")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title(f"Loss - {title}"); plt.legend()
    plt.tight_layout(); plt.show()
    b = int(np.argmin(hist["val_loss"]))
    print(f"Lowest val_loss {hist['val_loss'][b]:.4f} at epoch {b + 1}/{len(ep)} | val top-1 there {hist['val_accuracy'][b]:.4f} | val top-5 there {hist['val_top5'][b]:.4f}")

def fit_one(model, epochs, patience):
    return model.fit(train_generator, epochs=epochs, validation_data=validation_generator,
                     class_weight=class_weight, callbacks=get_callbacks(patience), verbose=1)

def train_simple(name, model, epochs, patience):
    t0 = time.time(); h = fit_one(model, epochs, patience)
    TRAIN_TIME[name] = time.time() - t0; HISTORIES[name] = h.history; trained[name] = model
    plot_history(h.history, name); print(f"Training time: {TRAIN_TIME[name] / 60:.1f} min")

### 4.1 Baseline — feed-forward (Dense) network on 64×64 images

In [ ]:
if RUN["baseline"]:
    baseline_model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)), layers.Resizing(64, 64), layers.Flatten(),
        layers.Dense(512, activation="relu"), layers.Dropout(0.4), layers.Dense(256, activation="relu"),
        layers.Dense(n_classes, activation="softmax")], name="baseline_dense")
    baseline_model.compile(loss=categorical_crossentropy, optimizer=Adam(1e-3), metrics=make_metrics())
    train_simple("Baseline (Dense)", baseline_model, EPOCHS["baseline"], 4)

### 💡 Insight
The Dense network flattens the picture, so it has **no notion of neighbouring pixels, edges or shapes**; it can only learn crude colour/texture statistics (e.g. 'dark coat', 'green background'). Expect accuracy only modestly above chance level and a big gap between training and validation — it is our *lower bound*.

### 4.2 CNN v1 — plain convolutional network (from scratch)

In [ ]:
if RUN["cnn1"]:
    cnn_model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(32, (3, 3), activation="relu"), layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"), layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu"), layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu"), layers.MaxPooling2D((2, 2)),
        layers.Flatten(), layers.Dense(256, activation="relu"), layers.Dense(n_classes, activation="softmax")], name="cnn_v1")
    cnn_model.compile(loss=categorical_crossentropy, optimizer=Adam(1e-3), metrics=make_metrics())
    train_simple("CNN v1", cnn_model, EPOCHS["cnn1"], 5)

### 💡 Insight
Convolutions share weights across the image and learn **local patterns** (edges → textures → ears/eyes), which is why CNNs normally beat the Dense net clearly. But with a few hundred images per class and no regularisation, a from-scratch CNN **memorises the training set**: watch training accuracy running far above validation accuracy and validation loss turning upwards.

### 4.3 CNN v2 — deeper and regularised (from scratch)
Two conv layers per block with `BatchNormalization`, progressive `Dropout`, `L2` weight decay and `GlobalAveragePooling2D` (instead of `Flatten`: far fewer parameters in the dense head).

In [ ]:
def conv_block(x, filters, drop):
    for _ in range(2):
        x = layers.Conv2D(filters, (3, 3), padding="same", use_bias=False, kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    return layers.Dropout(drop)(x)

if RUN["cnn2"]:
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)); x = inp
    for f, d in [(32, 0.10), (64, 0.15), (128, 0.20), (256, 0.25), (256, 0.30)]:
        x = conv_block(x, f, d)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    cnn_model_v2 = models.Model(inp, out, name="cnn_v2")
    cnn_model_v2.compile(loss=categorical_crossentropy, optimizer=Adam(1e-3), metrics=make_metrics())
    train_simple("CNN v2 (BN + Dropout + L2)", cnn_model_v2, EPOCHS["cnn2"], 8)

### 💡 Insight
Regularisation (BatchNorm, Dropout, L2, augmentation) **narrows the train–validation gap** and lowers validation loss compared with CNN v1, but a network trained from scratch on ~14 000 images still cannot learn what millions of ImageNet photos teach. It usually learns slowly at first (BatchNorm + Dropout need more epochs), so its curve may start flat.

### 4.4 Transfer learning — MobileNetV2 and EfficientNetB0
Both backbones were pretrained on **ImageNet** (which contains many dog breeds). Each is trained in two phases:
1. **Phase 1 — feature extraction:** backbone frozen, only the new classification head is trained (`lr = 1e-3`).
2. **Phase 2 — fine-tuning:** the top backbone layers are unfrozen and trained with a tiny learning rate (`lr = 2e-5`); BatchNorm layers stay frozen for stable fine-tuning.

The generators give pixels in `[0, 1]`; a `Rescaling` layer converts them to what each backbone expects (`[-1, 1]` for MobileNetV2, `[0, 255]` for EfficientNet). *Needs internet on the first run to download the weights.*

In [ ]:
USING_IMAGENET = {}
def build_transfer(name):
    ctor, pre = {"MobileNetV2": (MobileNetV2, layers.Rescaling(2.0, offset=-1.0)),
                 "EfficientNetB0": (EfficientNetB0, layers.Rescaling(255.0))}[name]
    try:
        base = ctor(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=PRETRAINED_WEIGHTS)
        USING_IMAGENET[name] = PRETRAINED_WEIGHTS == "imagenet"
    except Exception as e:
        print(f"WARNING: could not load ImageNet weights for {name} ({type(e).__name__}) - using RANDOM weights, results will be much worse.")
        base = ctor(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=None); USING_IMAGENET[name] = False
    base.trainable = False
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = pre(inp); x = base(x, training=False)               # BatchNorm stays in inference mode
    x = layers.GlobalAveragePooling2D()(x); x = layers.BatchNormalization()(x); x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x); x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    model = models.Model(inp, out, name=name.lower() + "_dogs")
    model.compile(loss=categorical_crossentropy, optimizer=Adam(1e-3), metrics=make_metrics())
    return model, base

def train_two_phase(name, model, base, unfreeze_last, epochs1, epochs2):
    t0 = time.time()
    h1 = fit_one(model, epochs1, 4)                                  # phase 1: head only
    base.trainable = True
    for layer in base.layers[:-unfreeze_last]: layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization): layer.trainable = False
    model.compile(loss=categorical_crossentropy, optimizer=Adam(2e-5), metrics=make_metrics())
    h2 = fit_one(model, epochs2, 6)                                  # phase 2: fine-tune
    hist = {k: h1.history[k] + h2.history[k] for k in h1.history if k in h2.history}
    TRAIN_TIME[name] = time.time() - t0; HISTORIES[name] = hist; trained[name] = model
    plot_history(hist, name + " (phase 1 + phase 2)")
    print(f"Phase 1 ran {len(h1.history['loss'])} epochs, phase 2 ran {len(h2.history['loss'])} epochs | training time {TRAIN_TIME[name] / 60:.1f} min")

#### CNN v3 — MobileNetV2

In [ ]:
if RUN["mobilenet"]:
    mobilenet_model, mobilenet_base = build_transfer("MobileNetV2")
    print("Pretrained ImageNet weights in use:", USING_IMAGENET["MobileNetV2"], "| parameters:", f"{mobilenet_model.count_params():,}")
    train_two_phase("CNN v3 (MobileNetV2)", mobilenet_model, mobilenet_base, 40, EPOCHS["mob1"], EPOCHS["mob2"])

#### CNN v4 — EfficientNetB0

In [ ]:
if RUN["effnet"]:
    effnet_model, effnet_base = build_transfer("EfficientNetB0")
    print("Pretrained ImageNet weights in use:", USING_IMAGENET["EfficientNetB0"], "| parameters:", f"{effnet_model.count_params():,}")
    train_two_phase("CNN v4 (EfficientNetB0)", effnet_model, effnet_base, 60, EPOCHS["eff1"], EPOCHS["eff2"])

### 💡 Insight
- **Phase 1** (frozen backbone) already gives a large jump: the pretrained features separate breeds well, and only a small head has to be learned. **Phase 2** (fine-tuning at a tiny learning rate) usually adds a few more points and lowers the validation loss further.
- In the merged curve, the point where phase 2 starts is visible as a change in slope. If validation loss rises during phase 2, `EarlyStopping` restores the best weights automatically.
- The two backbones trade **speed/size (MobileNetV2)** against **capacity (EfficientNetB0)**; the comparison below shows whether the extra capacity pays off on this data.

## 5. Model Comparison
Every model is scored on **train (un-augmented sample), validation and test** data at its restored best weights. The final model is chosen by **lowest validation loss** — the test set is not used for selection.

In [ ]:
assert trained, "No model was trained - check the RUN switches."
rows = []
for name, m in trained.items():
    tr = m.evaluate(train_eval_generator, verbose=0)      # [loss, top1, top5]
    va = m.evaluate(validation_generator, verbose=0)
    te = m.evaluate(test_generator, verbose=0)
    rows.append([name, m.count_params(), TRAIN_TIME[name] / 60, tr[1], va[1], te[1], va[2], te[2], va[0], te[0]])
comparison_df = pd.DataFrame(rows, columns=["Model", "Params", "Train min", "Train Acc", "Val Acc", "Test Acc (top-1)",
                                            "Val Top-5", "Test Top-5", "Val Loss", "Test Loss"])
comparison_df["Overfit gap (Train-Val)"] = comparison_df["Train Acc"] - comparison_df["Val Acc"]
display(comparison_df.round(4))

best_idx = int(comparison_df["Val Loss"].idxmin())
best_model_name = comparison_df.loc[best_idx, "Model"]
final_model = trained[best_model_name]
print("Best model (lowest validation loss):", best_model_name)

fig, ax = plt.subplots(1, 3, figsize=(20, 5)); x = np.arange(len(comparison_df)); w_ = 0.2
for k, (col, lab) in enumerate([("Train Acc", "Train"), ("Val Acc", "Validation"), ("Test Acc (top-1)", "Test top-1"), ("Test Top-5", "Test top-5")]):
    ax[0].bar(x + (k - 1.5) * w_, comparison_df[col], w_, label=lab)
ax[0].axhline(chance_top1, color="k", ls=":", label="chance top-1"); ax[0].set_xticks(x); ax[0].set_xticklabels(comparison_df["Model"], rotation=25, ha="right")
ax[0].set_ylim(0, 1.02); ax[0].set_title("Accuracy"); ax[0].legend(fontsize=8)
ax[1].bar(comparison_df["Model"], comparison_df["Val Loss"], color="indianred"); ax[1].set_title("Validation loss (lower is better)"); ax[1].tick_params(axis="x", rotation=25)
ax[2].bar(comparison_df["Model"], comparison_df["Overfit gap (Train-Val)"], color="slateblue"); ax[2].set_title("Overfitting gap: train - validation accuracy"); ax[2].tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

best = comparison_df.loc[best_idx]; weakest = comparison_df["Test Acc (top-1)"].min()
print(f"INSIGHT: best = {best_model_name}: test top-1 {best['Test Acc (top-1)']:.1%}, top-5 {best['Test Top-5']:.1%} "
      f"(chance: {chance_top1:.1%} / {chance_top5:.1%}); weakest model top-1 = {weakest:.1%}.")

### 💡 Insight — how to read the comparison
- **Top-1 vs top-5:** top-5 is always higher. A large gap means the model usually has the right breed *near the top* even when it is not first — typical for look-alike breeds.
- **Overfitting gap:** a large train–validation difference (right chart) marks memorisation. From-scratch models usually show the biggest gap; the pretrained models the smallest at the highest accuracy.
- **Parameters / time:** MobileNetV2 is the smallest and quickest; EfficientNetB0 has more capacity. If their accuracy is close, the smaller model is the more practical deployment choice.
- Selection is by **validation loss** (it rewards well-calibrated probabilities, not just correct arg-max), and the test set stays an independent check.

> ⚠️ **ImageNet caveat — read before quoting the transfer-learning numbers.** This dataset (ImageNet-style folder names, 120 breeds, ~20 000 images) appears to be a **subset of ImageNet** (the *Stanford Dogs* dataset). MobileNetV2/EfficientNetB0 were pretrained on ImageNet, so some of *our* validation/test photos may have been seen by the backbone during pretraining. Their accuracy here can therefore be **higher than on genuinely new photos**. The from-scratch models are not affected. To measure real-world accuracy, test the final model on **your own, new dog photos** (section 7).

## 6. Error Analysis of the Final Model

### 6.1 Overall metrics, confidence interval and the best / worst breeds
With 120 classes a full annotated confusion-matrix is unreadable, so we look at overall metrics with a **95 % confidence interval**, the **best and worst breeds** (F1), and how accuracy relates to class size.

In [ ]:
test_generator.reset()
y_prob = final_model.predict(test_generator, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_generator.classes
n = len(y_true)
top1 = (y_pred == y_true).mean()
rank = (y_prob > y_prob[np.arange(n), y_true][:, None]).sum(axis=1)       # 0 = true breed ranked first
top5 = (rank < min(5, n_classes)).mean()
z = 1.96; ctr = (top1 + z**2 / (2 * n)) / (1 + z**2 / n); half = z * np.sqrt(top1 * (1 - top1) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
print(f"Final model: {best_model_name}")
print(f"Test top-1 accuracy = {top1:.4f}  (95% CI {ctr - half:.3f} - {ctr + half:.3f}) on {n} images | top-5 accuracy = {top5:.4f}")

report = classification_report(y_true, y_pred, labels=np.arange(n_classes), target_names=class_labels, output_dict=True, zero_division=0)
rep_df = pd.DataFrame(report).T.iloc[:n_classes].sort_values("f1-score")
print(f"Macro F1 = {report['macro avg']['f1-score']:.4f} | weighted F1 = {report['weighted avg']['f1-score']:.4f}")
k = min(10, n_classes)
fig, ax = plt.subplots(1, 3, figsize=(20, 5.5))
rep_df.head(k)["f1-score"].iloc[::-1].plot(kind="barh", ax=ax[0], color="indianred"); ax[0].set_title(f"{k} hardest breeds (lowest F1)")
rep_df.tail(k)["f1-score"].plot(kind="barh", ax=ax[1], color="seagreen"); ax[1].set_title(f"{k} easiest breeds (highest F1)")
tr_size = train_data["label"].value_counts().reindex(class_labels).values
ax[2].scatter(tr_size, rep_df.reindex(class_labels)["recall"].values, alpha=.7, color="slateblue"); ax[2].set_xlabel("training images of the breed"); ax[2].set_ylabel("test recall"); ax[2].set_title("Does class size matter?")
plt.tight_layout(); plt.show()
corr = np.corrcoef(tr_size, rep_df.reindex(class_labels)["recall"].values)[0, 1] if len(set(tr_size)) > 1 else float("nan")
print(f"INSIGHT: correlation between a breed's training-set size and its test recall = {corr:+.2f}.")

### 💡 Insight
- The **confidence interval** shows how precise the headline accuracy is; differences between models smaller than that interval are not meaningful.
- **Hardest breeds** are typically look-alikes within a family or breeds with very variable coats; **easiest breeds** have a distinctive look. This tells you where more data (or a dedicated model) would help.
- If the scatter shows only a weak relationship, **class size is not the main driver of errors** — visual similarity between breeds is.

### 6.2 How close are the errors? (top-k curve, family-level errors, most-confused pairs)

In [ ]:
ks = np.arange(1, min(10, n_classes) + 1)
topk = [(rank < k_).mean() for k_ in ks]
cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted([(class_labels[i], class_labels[j], int(off[i, j])) for i, j in zip(*np.nonzero(off))], key=lambda t: -t[2])[:15]

wrong = np.where(y_pred != y_true)[0]
fam_arr = np.array([family_of(b) for b in class_labels])
same_family = np.mean([(fam_arr[y_true[i]] == fam_arr[y_pred[i]]) and fam_arr[y_true[i]] != "other" for i in wrong]) if len(wrong) else float("nan")

fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
ax[0].plot(ks, topk, marker="o", color="teal"); ax[0].set_xlabel("k"); ax[0].set_ylabel("share of images whose true breed is in the top-k"); ax[0].set_ylim(0, 1.02); ax[0].set_title("Top-k accuracy curve")
plt.sca(ax[1]); plt.imshow(cm / cm.sum(axis=1, keepdims=True).clip(min=1), cmap="Blues"); plt.colorbar(label="fraction of true breed")
plt.xlabel("predicted breed index"); plt.ylabel("true breed index"); plt.title("Confusion matrix (row-normalised)")
plt.tight_layout(); plt.show()
display(pd.DataFrame(pairs, columns=["True breed", "Predicted as", "Count"]))

fam_tab = pd.DataFrame({"family": fam_arr[y_true], "ok": (y_pred == y_true)}).groupby("family")["ok"].agg(["mean", "count"]).sort_values("mean")
display(fam_tab.rename(columns={"mean": "accuracy", "count": "test images"}).round(3))
print(f"INSIGHT: {len(wrong)} test errors; {same_family:.0%} of them stay inside the same breed family (e.g. terrier -> terrier). "
      f"Moving from top-1 to top-5 raises accuracy from {topk[0]:.1%} to {topk[min(4, len(topk) - 1)]:.1%}.")

### 💡 Insight
- The **top-k curve** rises steeply: most 'errors' are near-misses where the right breed is in the top few guesses. For a product, showing the **top-3/top-5 breeds** is much more useful than a single answer.
- The **family-level** result shows *what kind* of mistakes the model makes: a high share of errors inside the same family means the model understands dogs well but cannot resolve very fine differences — a much milder failure than random confusion.
- The **most-confused pairs** are usually the breeds that even humans mix up (similar terriers, spaniels, poodle sizes, retrievers).

### 6.3 Is the model's confidence trustworthy? (calibration)

In [ ]:
conf = y_prob.max(axis=1); correct = (y_pred == y_true)
bins = np.linspace(0, 1, 11); idx = np.clip(np.digitize(conf, bins) - 1, 0, 9)
cal = pd.DataFrame({"bin": idx, "conf": conf, "ok": correct}).groupby("bin").agg(mean_conf=("conf", "mean"), accuracy=("ok", "mean"), n=("ok", "size"))
ece = (cal["n"] / cal["n"].sum() * (cal["mean_conf"] - cal["accuracy"]).abs()).sum()

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].plot(cal["mean_conf"], cal["accuracy"], marker="o", color="crimson")
ax[0].set_xlabel("mean predicted confidence"); ax[0].set_ylabel("actual accuracy"); ax[0].set_title(f"Reliability diagram (ECE = {ece:.3f})")
ax[1].hist(conf[correct], bins=20, alpha=.7, label="correct", color="seagreen"); ax[1].hist(conf[~correct], bins=20, alpha=.7, label="wrong", color="indianred")
ax[1].set_xlabel("confidence of the top prediction"); ax[1].set_title("Confidence: correct vs wrong predictions"); ax[1].legend()
plt.tight_layout(); plt.show()
hi = conf >= 0.9
print(f"INSIGHT: {hi.mean():.0%} of test images get confidence >= 90%; their accuracy is {correct[hi].mean():.1%}." if hi.any() else "INSIGHT: no prediction reaches 90% confidence.")

### 💡 Insight
A model is **well calibrated** when its confidence matches its accuracy (points near the diagonal; small ECE). The right-hand histogram shows whether *wrong* predictions have lower confidence than correct ones — if so, a **confidence threshold** can be used in an application ('I am not sure — here are the top-3 breeds') to avoid presenting a doubtful guess as a fact.

### 6.4 Misclassified images (true breed vs predicted breed)

In [ ]:
show = wrong[:6]
if len(show):
    fig, axes = plt.subplots(2, len(show), figsize=(3.2 * len(show), 6.6)); axes = np.atleast_2d(axes).reshape(2, -1)
    for c_, i in enumerate(show):
        axes[0, c_].imshow(Image.open(test_data["image_path"][i]).convert("RGB")); axes[0, c_].axis("off")
        axes[0, c_].set_title(f"TRUE: {class_labels[y_true[i]]}\nPRED: {class_labels[y_pred[i]]} ({y_prob[i, y_pred[i]]:.2f})", fontsize=8)
        ref = train_data[train_data.label == class_labels[y_pred[i]]].sample(1, random_state=SEED)["image_path"].iloc[0]
        axes[1, c_].imshow(Image.open(ref).convert("RGB")); axes[1, c_].axis("off"); axes[1, c_].set_title(f"typical '{class_labels[y_pred[i]]}'", fontsize=8)
    plt.tight_layout(); plt.show()
print(f"{len(wrong)} misclassified out of {n}")

### 💡 Insight
The bottom row shows a typical photo of the **predicted** breed next to the misclassified image. In many cases the two breeds genuinely look alike, or the photo is hard (dog small in the frame, several dogs, heavy occlusion, unusual pose, low resolution). These cases define the ceiling of a *full-frame* classifier and motivate the next steps (dog detection + crop, higher resolution).

## 7. Prediction on a new, unseen image

In [ ]:
from tensorflow.keras.preprocessing import image

def predict_breed(img_path, model=None, top_k=5, show=True):
    # Same preprocessing as training: resize to IMG_SIZE x IMG_SIZE, scale pixels to 0-1.
    model = model or final_model
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = np.expand_dims(image.img_to_array(img) / 255.0, axis=0)
    p = model.predict(arr, verbose=0)[0]
    top = np.argsort(p)[::-1][:min(top_k, n_classes)]
    if show:
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ax[0].imshow(Image.open(img_path).convert("RGB")); ax[0].axis("off"); ax[0].set_title(f"Predicted: {class_labels[top[0]]} ({p[top[0]]:.1%})")
        ax[1].barh([class_labels[i] for i in top][::-1], p[top][::-1], color="steelblue"); ax[1].set_xlabel("probability"); ax[1].set_title("Top predictions")
        plt.tight_layout(); plt.show()
    return pd.DataFrame({"breed": [class_labels[i] for i in top], "probability": p[top]})

img_path = test_data["image_path"][0]          # <-- replace with your own dog photo, e.g. "my_dog.jpg"
print("True breed:", test_data["label"][0])
predict_breed(img_path)

### 💡 Insight
For a **real-world check** replace `img_path` with photos you took yourself (ideally of breeds from the list, in different settings). This is the most honest test of the whole system — in particular because of the ImageNet caveat in section 5. If the top-1 is wrong but the right breed appears in the top-5 list, the model behaves as the top-k analysis predicted.

## 8. Save the final model and verify that it reloads correctly

In [ ]:
final_model.save("dog_breed_classifier_final.keras")
json.dump(class_labels, open("dog_breed_labels.json", "w"))
json.dump({"model": best_model_name, "img_size": IMG_SIZE, "n_classes": n_classes, "preprocessing": "resize to img_size x img_size, pixels / 255",
           "test_top1": float(top1), "test_top5": float(top5), "dataset_url": DATA_URL}, open("dog_breed_meta.json", "w"), indent=2)
comparison_df.round(4).to_csv("model_comparison.csv", index=False)

reloaded = keras.models.load_model("dog_breed_classifier_final.keras")
xb, _ = test_generator[0]
same = np.allclose(final_model.predict(xb, verbose=0), reloaded.predict(xb, verbose=0), atol=1e-4)
print("Saved: dog_breed_classifier_final.keras, dog_breed_labels.json, dog_breed_meta.json, model_comparison.csv")
print("Reloaded model gives identical predictions:", same)

### 💡 Insight
The model, the **label list** (needed to turn output indices into breed names), a small metadata file (input size, preprocessing, measured accuracy) and the comparison table are saved together; the reload check proves the saved file is complete.

## 9. Conclusion
**What was done** — labelled the 120 breed folders, audited the data (corrupt files, sizes, near-duplicates, class balance, breed families), built leak-free train/validation/test sets, trained and compared five models (Dense baseline, CNN v1, regularised CNN v2, MobileNetV2, EfficientNetB0), selected the final model by **validation loss**, analysed its errors (best/worst breeds, confused pairs, family-level errors, calibration) and saved a verified model with its label list. All final numbers are those printed by *your* run; the notebook does not hard-code any result.

**Main lessons**
1. **Fine-grained classification with ~100–200 images per class needs transfer learning** — from-scratch CNNs overfit and plateau, pretrained backbones generalise far better.
2. **Top-5 (and the top-k curve) tells a better story than top-1**: most errors are near-misses between look-alike breeds of the same family.
3. **Measure more than accuracy**: per-breed F1, family-level errors and calibration show *where* and *how* the model fails and whether its confidence can be trusted.
4. **Be careful with benchmarks that overlap the pretraining data** (ImageNet caveat, section 5): validate on fresh photos before promising real-world accuracy.

**Limitations**
- Full-frame photos: background and people add noise; the dog is often small in the picture.
- ~20 test images per breed → per-breed numbers are coarse; one hold-out split (k-fold cross-validation would be more robust).
- Possible overlap between the pretraining data and this dataset (see section 5); the model only knows these 120 breeds (mixed breeds or other animals will still be forced into one of the 120).

**Next steps**
- **Detect and crop the dog** first (YOLO / SSD) before classification.
- Larger backbones (EfficientNetB3/V2, ConvNeXt) at higher resolution (300–384 px), **test-time augmentation** and an **ensemble** of MobileNetV2 + EfficientNetB0.
- **Grad-CAM** heat-maps to verify the model looks at the dog (not the background); mixup/cutmix; label smoothing.
- Add an **'unknown / not a dog' rejection** using the confidence threshold, and deploy behind a small API or web app returning the top-5 breeds with probabilities.